This is the beginning of our bronze data ingestion into DuckDB. Our first cell assigns our project root, raw data directory, database path, and the raw data file.

In [6]:
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "ltv.duckdb"

RAW_DIR.mkdir(parents=True, exist_ok=True)  

raw_file = RAW_DIR / "online_retail_raw_data.xlsx"
print(raw_file.exists())

True
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(3)
memory usage: 90.1+ MB


This cell imports DuckDB and pandas, and then concats the different sheets from our raw data into one table to use for our silver and gold notebooks. We also create our bronze table in our database.

In [ ]:
##run pkill -f ipykernel if you are getting a locking error

import duckdb
import pandas as pd

#Read every sheet (2009-2010 and 2010-2011) into a dict of DataFrames
sheets = pd.read_excel(raw_file, sheet_name=None, dtype={"Invoice": str, "StockCode": str})
for name, df in sheets.items():
    print(name, df.shape)

#Stack the sheets into one table (values unchanged)
bronze_df = pd.concat(sheets.values(), ignore_index=True)

con = duckdb.connect(str(DB_PATH))
con.execute("CREATE OR REPLACE TABLE bronze AS SELECT * FROM bronze_df")
print(con.sql("SELECT COUNT(*) AS rows FROM bronze").fetchone())
con.close()

Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)
(1067371,)


This cell is a spot check to ensure our bronze table has been created as expected.

In [3]:
with duckdb.connect(str(DB_PATH)) as con:
    display(con.sql("SHOW TABLES").df())

,name
0,bronze


In [4]:
# Summary of the table: number of rows, each column's name,
# how many non-missing values it has, and its data type
bronze_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(3)
memory usage: 65.1+ MB


In [5]:
bronze_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
